# INAI 6986 Milestone 1 — ALPHIN Reproducibility Notebook

This notebook reproduces the data-preparation, split, provenance, leakage, training-status, test-suite, and evaluation-evidence tables used in **Milestone 1 — Data and Model Selection**. It runs top to bottom from the downloaded official public-PDF snapshot and candidate records to deterministic curation and recomputation of diagnostic metrics from the saved per-prompt predictions.

**Model family and permission.** This project is in the **Deep Learning (DL)** family. With the course instructor's explicit permission, it fine-tunes a local Muse Glimmer 30B large language model (LLM) using quantized low-rank adaptation (QLoRA). The permission authorizes the modeling path; it does not waive the requirement for same-split baselines, candidate comparison, leakage checks, or honest evidence.

**Claim boundary.** The notebook does not rerun local LLM generation, training, or inference. It verifies the immutable inputs, deterministically rebuilds the ordinary-SFT artifact byte-for-byte, recomputes the reported diagnostics from all saved per-prompt outputs, and runs the repository tests. Training loss, successful model loading, and lexical diagnostics are not factual accuracy. The generated answer key came from the same local generation family as the training candidates. A separate the 30-record student-authored control set is included and remains reserved for separate human factual adjudication before a final factual-accuracy claim. Missing evidence is surfaced as an open gate, never replaced by an invented score.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib, importlib.util, json, os, platform, re, subprocess


def locate_repo_root():
    override = os.environ.get("ALPHIN_REPO_ROOT")
    candidates = ([Path(override).expanduser()] if override else []) + [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data/source/2026-09-14/source_manifest.jsonl").exists() and (candidate / "scripts/milestone1_data_prep.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Run from the repository or set ALPHIN_REPO_ROOT to a clone containing data/ and scripts/.")

ROOT = locate_repo_root()
print("Repository root located.")


## 1. Environment and Immutable Inputs

The notebook records the repository commit, runtime versions, and SHA-256 file fingerprints. All project paths are repository-relative, so a clone can be relocated without editing the notebook.


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]


def git(*args):
    return subprocess.run(["git", "-C", str(ROOT), *args], check=True, capture_output=True, text=True).stdout.strip()

FILES = {
    "source_manifest": ROOT / "data/source/2026-09-14/source_manifest.jsonl",
    "candidates": ROOT / "data/curated/2026-09-14/sft_train_candidates.jsonl",
    "ordinary_sft": ROOT / "artifacts/milestone1/2026-09-17/approved_candidates.jsonl",
    "quarantine": ROOT / "artifacts/milestone1/2026-09-17/quarantine.jsonl",
    "validation": ROOT / "artifacts/milestone1/2026-09-17/approved_validation.json",
    "training": ROOT / "artifacts/training/2026-09-17-muse-only-rerun.json",
    "progress": ROOT / "artifacts/milestone1/2026-09-22/progress_snapshot.json",
    "evaluation": ROOT / "artifacts/milestone1/2026-09-22/evaluation_snapshot.json",
}
EVALUATION_ROOT = ROOT / "artifacts/evaluation/2026-09-18"
missing = [str(p.relative_to(ROOT)) for p in FILES.values() if not p.exists()]
assert not missing, f"Missing required repository inputs: {missing}"
assert EVALUATION_ROOT.is_dir(), f"Missing raw evaluation artifact directory: {EVALUATION_ROOT.relative_to(ROOT)}"
contract = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "git_commit": git("rev-parse", "HEAD"),
    "git_branch": git("branch", "--show-current"),
    "input_sha256": {name: sha256(p) for name, p in FILES.items()},
}
contract


## 2. Raw Source Snapshot Verification

The source boundary is the immutable 2026-09-14 snapshot. After running the public-source download script, every one of the 138 PDFs is verified against its manifest fingerprint. The PDFs are not committed because the official manuals can contain published contact details; they are downloaded locally and remain untracked. Two reserved, one-page Administrative Code PDFs contain no candidate-bearing text, which explains why 138 source PDFs yield 136 candidate-bearing documents.


In [ ]:
sources = read_jsonl(FILES["source_manifest"])
source_root = ROOT / "data/source/2026-09-14"
pdf_by_name = defaultdict(list)
for pdf in source_root.rglob("*.pdf"):
    pdf_by_name[pdf.name].append(pdf)

source_errors = []
for source in sources:
    name = Path(source["local_path"]).name
    matches = pdf_by_name.get(name, [])
    if len(matches) != 1:
        source_errors.append({"file": name, "matches": len(matches)})
    elif sha256(matches[0]) != source["sha256"]:
        source_errors.append({"file": name, "error": "sha256 mismatch"})

assert len(sources) == 138
assert len(list(source_root.rglob("*.pdf"))) == 138
assert not source_errors, source_errors
source_verification = {
    "source_pdfs": len(sources),
    "source_pages": sum(int(s["pages"]) for s in sources),
    "manifest_sha256": sha256(FILES["source_manifest"]),
    "verification_errors": 0,
}
source_verification


## 3. Deterministic Curation, Quarantine, and Split

The notebook imports the production helper rather than reimplementing it. The helper removes the one blank-answer/unanswerable record from ordinary supervised fine-tuning (SFT), preserves it in quarantine for safety testing, and assigns whole documents to train, validation, or test through a repeatable SHA-256 rule.


In [ ]:
spec = importlib.util.spec_from_file_location("milestone1_data_prep", ROOT / "scripts/milestone1_data_prep.py")
prep = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prep)

candidates = read_jsonl(FILES["candidates"])
ordinary_saved = read_jsonl(FILES["ordinary_sft"])
quarantine_saved = read_jsonl(FILES["quarantine"])
ordinary_rebuilt, quarantine_rebuilt = prep.prepare_records(candidates)
summary_rebuilt = prep.summarize_records(ordinary_rebuilt)
validation = json.loads(FILES["validation"].read_text(encoding="utf-8"))

assert ordinary_rebuilt == ordinary_saved
assert quarantine_rebuilt == quarantine_saved
rebuilt_bytes = "".join(json.dumps(record, ensure_ascii=False) + "\n" for record in ordinary_rebuilt).encode("utf-8")
rebuilt_sha256 = hashlib.sha256(rebuilt_bytes).hexdigest()
assert rebuilt_bytes == FILES["ordinary_sft"].read_bytes()
assert rebuilt_sha256 == sha256(FILES["ordinary_sft"])
assert len(candidates) == 2000
assert len(ordinary_rebuilt) == 1999 and len(quarantine_rebuilt) == 1
assert summary_rebuilt["split_counts"] == {"train": 1684, "validation": 124, "test": 191}
assert summary_rebuilt["document_leakage_count"] == 0
assert validation["deterministic_pass"] is True and validation["blocking_issue_count"] == 0
assert validation["candidate_sha256"] == sha256(FILES["ordinary_sft"])

curation = {
    "candidate_records": len(candidates),
    "ordinary_sft_records": len(ordinary_rebuilt),
    "safety_quarantine_records": len(quarantine_rebuilt),
    "split_counts": summary_rebuilt["split_counts"],
    "candidate_bearing_documents": summary_rebuilt["unique_documents"],
    "document_overlap": summary_rebuilt["document_leakage_count"],
    "ordinary_sft_sha256": sha256(FILES["ordinary_sft"]),
    "rebuilt_byte_identical": rebuilt_bytes == FILES["ordinary_sft"].read_bytes(),
    "deterministic_blocking_issues": validation["blocking_issue_count"],
}
curation


## 4. Data Characteristics and Distribution

The target is a source-cited answer to a policy or provider-billing question—not an eligibility, payment, clinical, legal, or enforcement decision. Counts use one explicit definition: whitespace-delimited answer words. Source-family counts are joined through each file's SHA-256 fingerprint.


In [ ]:
source_family = {row["sha256"]: row["family"] for row in sources}
lengths = sorted(len(str(row.get("answer", "")).split()) for row in ordinary_rebuilt)


def percentile(values, p):
    position = (len(values) - 1) * p
    lower = int(position)
    upper = min(lower + 1, len(values) - 1)
    return values[lower] + (values[upper] - values[lower]) * (position - lower)


doc_split = {row["document_id"]: row["split"] for row in ordinary_rebuilt}
distribution = {
    "source_family_records": dict(Counter(source_family[row["source_sha256"]] for row in ordinary_rebuilt)),
    "documents_per_split": dict(Counter(doc_split.values())),
    "answer_words": {
        "minimum": lengths[0], "p25": percentile(lengths, .25), "median": percentile(lengths, .5),
        "p75": percentile(lengths, .75), "p90": percentile(lengths, .90),
        "p95": percentile(lengths, .95), "maximum": lengths[-1],
    },
    "answerable_ordinary_records": sum(not bool(row.get("unanswerable")) for row in ordinary_rebuilt),
    "unanswerable_safety_records": len(quarantine_rebuilt),
}
distribution


## 5. Exact and Near-Duplicate Leakage Audit

Document separation is necessary but not sufficient. This cell reports exact cross-split duplicate groups and deterministic candidate near-duplicates using token-set Jaccard similarity of at least 0.85. Near-duplicate findings are review candidates, not automatic proof of leakage.


In [ ]:
def normalize(text):
    return " ".join(re.findall(r"[a-z0-9]+", str(text).lower()))


def tokens(text):
    return frozenset(re.findall(r"[a-z0-9]+", str(text).lower()))


def exact_groups(field):
    groups = defaultdict(list)
    for row in ordinary_rebuilt:
        key = normalize(row.get(field, ""))
        if key:
            groups[key].append(row)
    duplicates = [group for group in groups.values() if len(group) > 1]
    cross = [group for group in duplicates if len({row["split"] for row in group}) > 1]
    return {"duplicate_groups": len(duplicates), "cross_split_groups": len(cross), "cross_split_records": sum(len(group) for group in cross)}


def near_duplicate_pairs(field="question", threshold=.85):
    row_tokens = [tokens(row.get(field, "")) for row in ordinary_rebuilt]
    index = defaultdict(list)
    for i, token_set in enumerate(row_tokens):
        for token in token_set:
            index[token].append(i)
    candidate_pairs = set()
    for members in index.values():
        if len(members) > 200:
            continue
        for pos, left in enumerate(members):
            for right in members[pos + 1:]:
                if ordinary_rebuilt[left]["split"] != ordinary_rebuilt[right]["split"]:
                    candidate_pairs.add((left, right))
    findings = []
    for left, right in candidate_pairs:
        a, b = row_tokens[left], row_tokens[right]
        if not a or not b or a == b:
            continue
        score = len(a & b) / len(a | b)
        if score >= threshold:
            findings.append({
                "left": ordinary_rebuilt[left]["prompt_id"], "right": ordinary_rebuilt[right]["prompt_id"],
                "left_split": ordinary_rebuilt[left]["split"], "right_split": ordinary_rebuilt[right]["split"],
                "jaccard": round(score, 4),
            })
    return sorted(findings, key=lambda item: (-item["jaccard"], item["left"], item["right"]))


near_question_pairs = near_duplicate_pairs("question")
leakage = {
    "document_overlap": 0,
    "exact_questions": exact_groups("question"),
    "exact_answers": exact_groups("answer"),
    "near_question_cross_split_candidates_at_0_85": len(near_question_pairs),
    "sample_near_question_pairs": near_question_pairs[:10],
}
leakage


## 6. Answer-Key Provenance and Independent Control

All 191 generated test records came from the same local Muse generation family that produced the training candidates: 100 from `muse-glimmer-30b` and 91 from `muse-glimmer-30b-local-endpoint-2`. Document separation prevents passage overlap but cannot prevent the adapter from learning the pipeline's wording and answer conventions.

The professor-feedback requirement is met by the **30-record student-authored control set** in `data/curated/student-authored-control-set-2026-09-22.jsonl`. It was created directly from source PDFs, kept out of generation and training, and is reserved for separate post-freeze human scoring.


In [ ]:
test_rows = [row for row in ordinary_rebuilt if row["split"] == "test"]
control_path = ROOT / "data/curated/student-authored-control-set-2026-09-22.jsonl"
control_rows = read_jsonl(control_path) if control_path.exists() else []
provenance = {
    "generated_test_records": len(test_rows),
    "generated_test_models": dict(Counter(row["generator_model"] for row in test_rows)),
    "student_authored_control_path": str(control_path.relative_to(ROOT)),
    "student_authored_control_records": len(control_rows),
    "required_control_range": "30-50",
    "control_gate_pass": 30 <= len(control_rows) <= 50,
}
provenance


## 7. Training and Model-Development Progress

The rejected 2026-09-16 mixed-generator run is procedural evidence only. The canonical clean candidate is the 2026-09-17 all-Muse corrective run on the 1,684/124/191 split with the blank case quarantined. A second all-Muse run finished September 18, but its operational 1,685-row render is not substituted for the canonical clean-data evidence run.


In [ ]:
training = json.loads(FILES["training"].read_text(encoding="utf-8"))
progress = json.loads(FILES["progress"].read_text(encoding="utf-8"))
evaluation = json.loads(FILES["evaluation"].read_text(encoding="utf-8"))
assert training["dataset_counts"] == {"train": 1684, "validation": 124, "test": 191}
assert training["adapter_sha256"] == "b27cca007157b3c1efcdfe4e7798ca49cf159d2a12fad0eeafe1349faef3cac0"
print("Evidence cutoff:", progress["evidence_cutoff"])
progress["training"]


## 8. Same-Split Candidate Diagnostics — Recomputed, Not Factual Accuracy

The completed 512-token run asked the same 191 generated test prompts of untouched Muse and two adapters. This section reads every saved per-prompt output, verifies its SHA-256 fingerprint, recomputes each aggregate diagnostic, and asserts agreement with the immutable summaries and report snapshot. It does not rerun inference. Answer-token F1, numeric exact match, citation presence, truncation, and the one-record abstention check are useful engineering diagnostics; they are **not** the committed factual-accuracy metric. The formal comparison blocked promotion because outputs were truncated and the clean adapter failed the one-record abstention check; no winner was selected.


In [ ]:
assert evaluation["test_set_sha256"] == "89e747acfef5dba7bb6b2bf14b89ca8602f817f0abdb85a9a06e9f392d2cd334"
assert all(row["records"] == 191 for row in evaluation["systems"])


def rounded_mean(rows, key):
    return round(sum(float(row[key]) for row in rows) / len(rows), 6)


def recompute_diagnostics(rows, abstention_rows):
    numeric_rows = [row for row in rows if row["numeric_reference_present"]]
    return {
        "records": len(rows),
        "answer_token_f1": rounded_mean(rows, "answer_token_f1"),
        "numeric_exact_match_rate": rounded_mean(numeric_rows, "numeric_exact_match"),
        "citation_url_rate": rounded_mean(rows, "citation_url_present"),
        "citation_page_rate": rounded_mean(rows, "citation_page_present"),
        "strict_citation_rate": rounded_mean(rows, "citation_present"),
        "truncated_rate": rounded_mean(rows, "generation_truncated"),
        "abstention_accuracy": rounded_mean(abstention_rows, "abstention_correct"),
        "abstention_n": len(abstention_rows),
    }

snapshot_by_label = {row["label"]: row for row in evaluation["systems"]}
recomputed_by_label = {}
raw_hashes = {}
for artifact in evaluation["raw_evaluation_artifacts"]:
    label = artifact["label"]
    ordinary_path = ROOT / artifact["evaluation_path"]
    abstention_path = ROOT / artifact["abstention_path"]
    summary_path = EVALUATION_ROOT / label / "evaluation.summary.json"
    abstention_summary_path = EVALUATION_ROOT / label / "abstention.summary.json"
    assert sha256(ordinary_path) == artifact["evaluation_sha256"]
    assert sha256(abstention_path) == artifact["abstention_sha256"]
    assert sha256(summary_path) == snapshot_by_label[label]["summary_sha256"]
    assert sha256(abstention_summary_path) == snapshot_by_label[label]["abstention_summary_sha256"]
    rows = read_jsonl(ordinary_path)
    abstention_rows = read_jsonl(abstention_path)
    assert len({row["prompt_id"] for row in rows}) == 191
    assert {row["input_hash"] for row in rows} == {evaluation["test_set_sha256"]}
    assert {row["prompt_ids_sha256"] for row in rows} == {evaluation["prompt_ids_sha256"]}
    recomputed = recompute_diagnostics(rows, abstention_rows)
    for metric, value in recomputed.items():
        assert value == snapshot_by_label[label][metric], (label, metric, value, snapshot_by_label[label][metric])
    recomputed_by_label[label] = recomputed
    raw_hashes[label] = {
        "evaluation_sha256": artifact["evaluation_sha256"],
        "abstention_sha256": artifact["abstention_sha256"],
    }

comparison_path = ROOT / evaluation["formal_comparison"]["comparison_path"]
assert sha256(comparison_path) == evaluation["formal_comparison"]["comparison_sha256"]
formal_comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
assert formal_comparison["pass"] is False
assert formal_comparison["paired_deltas"]["paired_records"] == 191
assert formal_comparison["paired_deltas"]["mean_deltas"]["answer_token_f1"] == evaluation["formal_comparison"]["answer_token_f1_delta"]
assert formal_comparison["paired_deltas"]["mean_deltas"]["citation_url_present"] == evaluation["formal_comparison"]["citation_url_presence_delta"]

diagnostic_table = []
for row in evaluation["systems"]:
    recomputed = recomputed_by_label[row["label"]]
    diagnostic_table.append({
        "system": row["system"],
        "n": recomputed["records"],
        "factual_accuracy": None,
        "answer_token_f1": recomputed["answer_token_f1"],
        "numeric_exact_match_rate": recomputed["numeric_exact_match_rate"],
        "citation_url_rate": recomputed["citation_url_rate"],
        "truncated_rate": recomputed["truncated_rate"],
        "abstention_accuracy": recomputed["abstention_accuracy"],
        "abstention_n": recomputed["abstention_n"],
    })
print("Formal blocked reasons:", evaluation["formal_comparison"]["blocked_reasons"])
print("Raw per-prompt artifact hashes verified for:", sorted(raw_hashes))
diagnostic_table


## 9. Rubric Metric Table and Completion Gates

The committed primary metric remains human-adjudicated factual accuracy on answerable cases. Secondary metrics reflect the asymmetric risk of unsupported Medicaid answers. `None` is a deliberate fail-closed value, not a zero and not permission to infer success from token overlap.


In [ ]:
metric_table = [
    {"system": "Manual current practice", "locked_n": None, "factual_accuracy": None, "locator_accuracy": None, "unsupported_claim_rate": None, "abstention": None, "latency_seconds": None},
    {"system": "Untouched Muse Glimmer 30B", "locked_n": 191, "factual_accuracy": None, "locator_accuracy": None, "unsupported_claim_rate": None, "abstention": None, "latency_seconds": None},
    {"system": "Muse QLoRA — 2026-09-17 clean adapter", "locked_n": 191, "factual_accuracy": None, "locator_accuracy": None, "unsupported_claim_rate": None, "abstention": None, "latency_seconds": None},
    {"system": "Muse QLoRA — 2026-09-18 adapter", "locked_n": 191, "factual_accuracy": None, "locator_accuracy": None, "unsupported_claim_rate": None, "abstention": None, "latency_seconds": None},
]
completion_gates = {
    "same_frozen_split": True,
    "document_overlap_zero": leakage["document_overlap"] == 0,
    "student_authored_control_30_to_50": provenance["control_gate_pass"],
    "human_factual_scores_present": all(row["factual_accuracy"] is not None for row in metric_table),
    "manual_baseline_present": metric_table[0]["factual_accuracy"] is not None,
    "base_adapter_retrieval_2x2_present": False,
    "measured_interpretability_ablations_present": False,
}
print("Metric table:")
print(json.dumps(metric_table, indent=2))
print("\nCompletion gates:")
completion_gates


## 10. Selection Logic, Null Result, and Dated Fallback

Muse Glimmer 30B with QLoRA remains a **provisional** candidate because it fits a text-only corpus, local/private deployment, 4-bit training feasibility, answer-only Muse ATEM formatting, rollback through a small adapter, and the asymmetric cost of unsupported answers. It is not a proven performance winner until the locked factual evaluation is complete.

If base-plus-retrieval and ALPHIN-plus-retrieval perform similarly, the conclusion will be that retrieval supplied most of the measurable benefit and domain adaptation added little once grounding was available. That is an anticipated, decision-relevant finding.


In [ ]:
print(progress["fallback_trigger"])
print("\nOpen gates at the evidence cutoff:")
for number, gate in enumerate(progress["open_gates"], 1):
    print(f"{number}. {gate}")


## 11. Repository Test-Suite Verification

The notebook runs the repository's default test target in the same top-to-bottom execution. The command is explicit and its stdout is saved below; a nonzero result stops the notebook.


In [ ]:
test_process = subprocess.run(
    ["uv", "run", "--with", "pytest", "pytest", "-q"],
    cwd=ROOT,
    capture_output=True,
    text=True,
)
print(test_process.stdout.strip())
if test_process.stderr.strip():
    print(test_process.stderr.strip())
assert test_process.returncode == 0, f"pytest failed with code {test_process.returncode}"
test_verification = {
    "command": "uv run --with pytest pytest -q",
    "returncode": test_process.returncode,
    "summary": test_process.stdout.strip().splitlines()[-1],
}


## 12. Final Evidence Ledger

This cell emits the exact counts, fingerprints, repository commit, training status, independently recomputed diagnostic boundary, test status, and unresolved gates for synchronization with the DOCX/PDF. A report may cite these values only if the notebook runs without assertion failure.


In [ ]:
final_ledger = {
    "repository": progress["source_of_record"]["repository"],
    "notebook_url": "https://github.com/porespellar/INAI6986_project_deliverables/blob/main/notebook/milestone1_reproducibility.ipynb",
    "git_commit_executed": contract["git_commit"],
    "evidence_cutoff": progress["evidence_cutoff"],
    "sources": source_verification,
    "data": curation,
    "distribution": distribution,
    "leakage": leakage,
    "answer_key_provenance": provenance,
    "clean_training_run": {
        "finished_at": training["finished_at"],
        "steps": training["global_step"],
        "train_loss": training["train_loss"],
        "final_eval_loss": training["final_eval_loss"],
        "adapter_sha256": training["adapter_sha256"],
        "interpretation": "training/procedural evidence; not factual accuracy",
    },
    "diagnostic_evaluation_snapshot": evaluation,
    "raw_evaluation_hashes": raw_hashes,
    "recomputed_diagnostic_table": diagnostic_table,
    "test_verification": test_verification,
    "rubric_metric_table": metric_table,
    "completion_gates": completion_gates,
    "fallback_trigger": progress["fallback_trigger"],
}
print(json.dumps(final_ledger, indent=2))


## Interpretation

After downloading the official public-PDF snapshot with the provided script, this notebook runs top to bottom from that source snapshot and the generated candidate records to deterministic byte-identical curation, split/distribution/leakage evidence, immutable training status, repository tests, and diagnostics recomputed from all saved per-prompt predictions. It intentionally does **not** rerun local LLM candidate generation, training, or inference. The student-authored control is complete and documented, while human factual scoring, the manual baseline, retrieval/no-retrieval comparison, and measured model ablations remain open. Those gates must be completed and the notebook rerun before a 100/100-ready submission can honestly contain the rubric's required primary-metric comparison.
